In [1]:
import pandas as pd
import numpy as np
import re

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [2]:
# Step 1: Load Dataset

df = pd.read_csv('sentiment_data.csv')

# Drop unwanted index column
df = df.drop(columns=['Unnamed: 0'])

print("Dataset Head:\n", df.head())
print("\nMissing values:\n", df.isnull().sum())

Dataset Head:
                                              Comment  Sentiment
0  lets forget apple pay required brand new iphon...          1
1  nz retailers don’t even contactless credit car...          0
2  forever acknowledge channel help lessons ideas...          2
3  whenever go place doesn’t take apple pay doesn...          0
4  apple pay convenient secure easy use used kore...          2

Missing values:
 Comment      217
Sentiment      0
dtype: int64


In [3]:
# Step 2: Handle Missing Values

df['Comment'] = df['Comment'].astype(str)   # Convert to string
df = df[df['Comment'].str.strip() != ""]   # Remove empty comments



In [6]:
# Step 3: Text Preprocessing

def preprocess_text(text):
    # handle missing / non-string values safely
    if pd.isna(text):
        return ""
    if not isinstance(text, str):
        text = str(text)
    text = text.lower()                        # lowercase
    text = re.sub(r'http\S+', '', text)        # remove URLs
    text = re.sub(r'[^a-zA-Z\s]', '', text)    # remove special characters & numbers
    text = re.sub(r'\s+', ' ', text).strip()   # remove extra spaces
    return text

df['clean_comment'] = df['Comment'].apply(preprocess_text)

print("\nSample cleaned text:\n")
print(df[['Comment', 'clean_comment']].head())




Sample cleaned text:

                                             Comment  \
0  lets forget apple pay required brand new iphon...   
1  nz retailers don’t even contactless credit car...   
2  forever acknowledge channel help lessons ideas...   
3  whenever go place doesn’t take apple pay doesn...   
4  apple pay convenient secure easy use used kore...   

                                       clean_comment  
0  lets forget apple pay required brand new iphon...  
1  nz retailers dont even contactless credit card...  
2  forever acknowledge channel help lessons ideas...  
3  whenever go place doesnt take apple pay doesnt...  
4  apple pay convenient secure easy use used kore...  


In [7]:
# Step 4: Convert Text to Features (TF-IDF)

tfidf = TfidfVectorizer(max_features=5000, stop_words='english')

X = tfidf.fit_transform(df['clean_comment'])
y = df['Sentiment']



In [8]:
# Step 5: Train-Test Split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)



In [9]:
# Step 6: Train Decision Tree Classifier

dt_model = DecisionTreeClassifier(
    criterion='gini',
    max_depth=25,
    random_state=42
)

dt_model.fit(X_train, y_train)



,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.",'gini'
,"splitter splitter: {""best"", ""random""}, default=""best""The strategy used to choose the split at each node. Supportedstrategies are ""best"" to choose the best split and ""random"" to choosethe best random split.",'best'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",25
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: int, float or {""sqrt"", ""log2""}, default=NoneThe number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... note:: The search for a split does not stop until at least one valid partition of the node samples is found, even if it requires to effectively inspect more than ``max_features`` features.",None
,"random_state random_state: int, RandomState instance or None, default=NoneControls the randomness of the estimator. The features are alwaysrandomly permuted at each split, even if ``splitter`` is set to``""best""``. When ``max_features < n_features``, the algorithm willselect ``max_features`` at random at each split before finding the bestsplit among them. But the best found split may vary across differentruns, even if ``max_features=n_features``. That is the case, if theimprovement of the criterion is identical for several splits and onesplit has to be selected at random. To obtain a deterministic behaviourduring fitting, ``random_state`` has to be fixed to an integer.See :term:`Glossary ` for details.",42
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow a tree with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current n

In [10]:
# Step 7: Prediction & Evaluation

y_pred = dt_model.predict(X_test)

print("\nAccuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))


Accuracy: 0.5947251653569429

Classification Report:
               precision    recall  f1-score   support

           0       0.79      0.22      0.35     11023
           1       0.50      0.86      0.63     16594
           2       0.72      0.58      0.64     20612

    accuracy                           0.59     48229
   macro avg       0.67      0.55      0.54     48229
weighted avg       0.66      0.59      0.57     48229


Confusion Matrix:
 [[ 2446  6142  2435]
 [  185 14255  2154]
 [  455  8175 11982]]


In [11]:
# Step 8: Test on New Comment

def predict_sentiment(text):
    text = preprocess_text(text)
    text_vec = tfidf.transform([text])
    pred = dt_model.predict(text_vec)[0]
    return pred

print("\nTest Examples:")
print("Prediction:", predict_sentiment("Apple Pay is very convenient"))
print("Prediction:", predict_sentiment("This service is very bad"))
print("Prediction:", predict_sentiment("Today is very good day"))
print("Prediction:", predict_sentiment("This service is not good"))




Test Examples:
Prediction: 1
Prediction: 0
Prediction: 2
Prediction: 2
